# 応用編5
このノートブックでは、プロキシ環境下にあるクライアントからブロックチェーンにアクセスする方法を示します。

## 概要
BC+では、クライアントからブロックチェーンにアクセスするプロトコルはhttpsです。  
したがって、プロキシ経由でhttps接続することも可能になっています。  
ブラウザからアクセスする場合には、ブラウザのプロキシの設定をするだけでよく、特段のプログラミングは必要ありません。  
ここでは、Node.jsからアクセスする場合のプロキシ対応方法について説明します。  


## 事前の準備

このノートブックを実行する前に、以下のコマンドを実行しておいてください。

npm install https-proxy-agent

## 準備

Node.js用のクライアントライブラリを読み込みます。  
（他のサンプルコードで利用しているlib/load-blockchain-api.mjsにはすでにプロキシ対応が組み込まれているため、ここでは使用しません）

In [1]:
var { default: api } = await import('../lib/dncware-blockchain-nodejs-async-api.cjs');

プロキシの設定が（あれば）読み出します。

In [2]:
var { proxy, CA } = await import('../lib/load-config.mjs');

プロキシ環境下にない場合、ローカルにプロキシを立ち上げ、プロキシ環境下と同様にします。

In [3]:
if(!proxy){
    var url = await import('node:url');
    var net = await import('node:net');
    var http = await import('node:http');
    var server = http.createServer();
    server.on('error', console.error);
    server.on('connect', function(req, sock){
        var u = url.parse("http://" + req.url);
        var sock2 = net.connect(u.port||443, u.hostname, function() {
            sock.write("HTTP/1.1 200 Connection established\r\n\r\n");
            sock.pipe(sock2);
            sock2.pipe(sock);
        });
        sock2.on('error', e => sock.destroy());
    });
    server.listen(8808);
    server.unref();
    var proxy = "http://localhost:8808";
}

その他の設定を読み出します。

In [4]:
var { chainID, peerURL } = await import('../lib/load-config.mjs');

## プロキシを設定

プロキシのagentを作成します。

In [5]:
var { HttpsProxyAgent } = await import('https-proxy-agent');
var agent = new HttpsProxyAgent(proxy);

CAの設定が必要な場合には読み込みます。

In [6]:
if(CA){
    var fs = await import('node:fs');
    var path = await import('node:path');
    var { default: package_root } = await import('../lib/get-package-root.mjs');
    var ca = fs.readFileSync(path.resolve(package_root, 'etc', CA));
}

ブロックチェーンに接続します。  
rpc.connectのオプションに、プロキシの設定を渡します。

In [7]:
var rpc = new api.RPC(chainID);
rpc.connect(peerURL, { agent, ca });

SocketHTTP {
  _triggers: Set(0) {},
  url: 'https://trial1.dncware-blockchain.biz',
  options: {
    agent: HttpsProxyAgent {
      _events: [Object: null prototype] {
        free: [Function (anonymous)],
        newListener: [Function: maybeEnableKeylog]
      },
      _eventsCount: 2,
      _maxListeners: undefined,
      options: { path: undefined },
      requests: [Object: null prototype] {},
      sockets: [Object: null prototype] {},
      freeSockets: [Object: null prototype] {},
      keepAliveMsecs: 1000,
      keepAlive: false,
      maxSockets: Infinity,
      maxFreeSockets: 256,
      scheduling: 'lifo',
      maxTotalSockets: Infinity,
      totalSocketCount: 0,
      proxy: URL {
        href: 'http://localhost:8808/',
        origin: 'http://localhost:8808',
        protocol: 'http:',
        username: '',
        password: '',
        host: 'localhost:8808',
        hostname: 'localhost',
        port: '8808',
        pathname: '/',
        search: '',
        searc

## ブロックチェーンにアクセス

プロキシの設定後は、通常の場合と同様にしてブロックチェーンにアクセスできます。

In [8]:
var wallet = await api.importSigningWallet('es', await api.generateWalletKey('es')); // anonymous wallet
var resp = await rpc.call(wallet, 'c1query', { type: 'dashboard' });
console.log(resp);

{
  txid: 'xW9nExRU6g8Am73yuG2JRkHSR7U8EBKACbnDutqcwNWJRB',
  status: 'ok',
  value: {
    N: 3,
    F: 1,
    B: 0,
    num_transactions: 702014,
    num_blocks: 73538,
    num_users: 46948,
    num_contracts: 907,
    num_groups: 147,
    num_domains: 169
  }
}
